In [1]:
import requests
import os
import json
import time
import pandas as pd
import sys
import os
import numpy as np
from datetime import date

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

import src.modify_reps
import src.gen_committees

In [2]:
try: 
    df = pd.read_json(os.path.join(project_root, "src", "generated_outputs", "congressmen.json"))
except Exception as e:
    print("There is an issue with the congressmen.json. Quitting.")
    sys.exit()

try: 
    vote_df = pd.read_json(os.path.join(project_root, "src", "generated_outputs", "vote_avg.json"))
except Exception as e:
    print("There is an issue with the vote_avg.json. Quitting.")
    sys.exit()

In [3]:
df[df['bioguideID']=="W000194"]

,bioguideID,name,partyName,state,url,attribution,imageUrl,chamber,endYear,startYear
489,W000194,"Watkins, Wes",Republican,Oklahoma,https://api.congress.gov/v3/member/W000194?for...,Collection of the U.S. House of Representatives,https://www.congress.gov/img/member/w000194_20...,House of Representatives,1991.0,1977
490,W000194,"Watkins, Wes",Republican,Oklahoma,https://api.congress.gov/v3/member/W000194?for...,Collection of the U.S. House of Representatives,https://www.congress.gov/img/member/w000194_20...,House of Representatives,2003.0,1997


Run all the modify_reps.py steps

In [4]:
debug = 0

df = src.modify_reps.update_endyear(df)
df = src.modify_reps.add_tenure(df)
df = src.modify_reps.normalize_name(df)
df = src.modify_reps.only_current(df)
df = src.modify_reps.merge_in_voting_records(df, vote_df)
df = src.modify_reps.replace_democratic(df)

#Functions you need to do on merged vote and reps:
df = src.modify_reps.get_voter_rank(df)

comm_dict = src.gen_committees.gen_committees()
df = src.modify_reps.merge_in_comms(df, comm_dict)




#df[df['bioguideID']=='W000194']


2025-12-11 15:45:37,294 - gen_committees.py - WARNING - Comcode not found for P000197, Nonetype returned.
2025-12-11 15:45:37,295 - gen_committees.py - WARNING - Comcode not found for S001176, Nonetype returned.
2025-12-11 15:45:37,295 - gen_committees.py - WARNING - Comcode not found for J000299, Nonetype returned.
2025-12-11 15:45:37,296 - gen_committees.py - WARNING - Comcode not found for C001101, Nonetype returned.
2025-12-11 15:45:37,297 - gen_committees.py - WARNING - Comcode not found for None, Nonetype returned.
2025-12-11 15:45:37,298 - gen_committees.py - WARNING - Comcode not found for J000294, Nonetype returned.
2025-12-11 15:45:37,298 - gen_committees.py - WARNING - Comcode not found for V000139, Nonetype returned.


Now for some stats, for your reference:

In [5]:
print(comm_dict)

{'B001323': ['Committee on Natural Resources', 'Committee on Transportation and Infrastructure', 'Committee on Science, Space, and Technology', 'Vice Chair: Committee on Natural Resources: Energy and Mineral Resources', 'Committee on Natural Resources: Oversight and Investigations', 'Committee on Transportation and Infrastructure: Aviation', 'Committee on Transportation and Infrastructure: Coast Guard and Maritime Transportation', 'Committee on Transportation and Infrastructure: Railroads, Pipelines, and Hazardous Materials', 'Committee on Science, Space, and Technology: Environment', 'Committee on Science, Space, and Technology: Energy', 'Committee on Science, Space, and Technology: Investigations and Oversight'], 'M001212': ['Committee on Agriculture', 'Committee on the Judiciary', 'Vice Chair: Committee on Agriculture: Forestry and Horticulture', 'Committee on Agriculture: General Farm Commodities, Risk Management, and Credit', 'Committee on Agriculture: Livestock, Dairy, and Poultr

In [6]:

print(f"There are {len(df[df['current_member']=="yes"])} current members")

#df.sort_values(by='tenure_current', ascending=True).head()


There are 537 current members


Now you can start piloting your definitions here.

In [15]:
#get average vote for reps in house, reps in senate, dems in house, dems in senate

avg_vote = df.groupby(['chamber', 'partyName'])[['with_D_percent', 'with_R_percent']].mean()
print(avg_vote)

avg_vote = avg_vote.reset_index()
avg_vote['dummy'] = 1

avg_vote_pivot = avg_vote.pivot_table(
    index='dummy', # Use a dummy index since we want all data in one row
    columns=['chamber', 'partyName'],
    values=['with_D_percent', 'with_R_percent']
)

avg_vote_pivot = avg_vote_pivot.reset_index(drop=True)
avg_vote_pivot.columns = ['_'.join(map(str, col)).replace(' ', '_') for col in avg_vote_pivot.columns]

avg_vote_pivot.rename(columns=lambda x: x.replace('House_of_Representatives', 'avg_vote_H')
                                .replace('Senate', 'avg_vote_S')
                                .replace('_Democrat', '_D')
                                .replace('_Republican', '_R')
                                .replace('_Independent', '_I')
                                .replace('with_D_percent', 'with_D')
                                .replace('with_R_percent', 'with_R')
                                .replace('0_', '', 1) # Remove the dummy index column name part
                                , inplace=True)

avg_vote_pivot = avg_vote_pivot.reset_index(drop=True)

print(avg_vote_pivot)




                                      with_D_percent  with_R_percent
chamber                  partyName                                  
House of Representatives Democrat          95.902326        4.097674
                         Republican         2.914414       97.085586
Senate                   Democrat               94.6             5.4
                         Independent            94.0             6.0
                         Republican         0.943396       99.056604
   with_D_avg_vote_H_D  with_D_avg_vote_H_R  with_D_avg_vote_S_D  \
0            95.902326             2.914414                 94.6   

   with_D_avg_vote_S_I  with_D_avg_vote_S_R  with_R_avg_vote_H_D  \
0                 94.0             0.943396             4.097674   

   with_R_avg_vote_H_R  with_R_avg_vote_S_D  with_R_avg_vote_S_I  \
0            97.085586                  5.4                  6.0   

   with_R_avg_vote_S_R  
0            99.056604  


chamber                   partyName  
House of Representatives  Democrat       46
                          Republican     46
Senate                    Democrat       36
                          Independent    24
                          Republican     48
Name: duration, dtype: int64


   max_tenure_H_D  max_tenure_H_R  max_tenure_S_D  max_tenure_S_I  \
0            46.0            46.0            36.0            24.0   

   max_tenure_S_R  
0            48.0  


   count_D  count_I  count_R
0      260        2      275


            count_D  count_I  count_R
bioguideID      260        2      275


                   0
max_tenure_H_D  46.0
max_tenure_H_R  46.0
max_tenure_S_D  36.0
max_tenure_S_I  24.0
max_tenure_S_R  48.0
count_D          260
count_I            2
count_R          275
max_tenure_H_I   NaN


In [9]:
df_current = df[df['current_member']=="yes"]

df_current[df_current['committees'].isna()]

,bioguideID,name,partyName,state,url,attribution,imageUrl,chamber,endYear,startYear,...,Abstained,Both,Neither,with_D,with_R,vote_count,with_party_count,with_party_percent,with_party_rank,committees
